In [2]:
!pip install pandas

  Using cached pandas-3.0.3-cp314-cp314-manylinux_2_24_x86_64.manylinux_2_28_x86_64.whl.metadata (79 kB)
  Using cached numpy-2.4.6-cp314-cp314-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (6.6 kB)
Using cached pandas-3.0.3-cp314-cp314-manylinux_2_24_x86_64.manylinux_2_28_x86_64.whl (10.9 MB)
Using cached numpy-2.4.6-cp314-cp314-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl (16.6 MB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [pandas]━━━━ 1/2 [pandas]


In [3]:
import pandas as pd

jobs = pd.read_csv('jobs.csv')
candidates = pd.read_csv('candidates.csv')
applications_train = pd.read_csv('applications_train.csv')

print("--- 1. Unique Values ---")
print("Currencies:", jobs['salary_currency'].dropna().unique())
print("English Levels:", candidates['english_proficiency'].dropna().unique())
print("Education Levels:", candidates['education_level'].dropna().unique())
print("\n")

print("--- 2. Missing Values ---")
print("[Jobs]\n", jobs.isnull().sum()[jobs.isnull().sum() > 0])
print("\n[Candidates]\n", candidates.isnull().sum()[candidates.isnull().sum() > 0])
print("\n[Train]\n", applications_train.isnull().sum()[applications_train.isnull().sum() > 0])
print("\n")

print("--- 3. Data Types ---")
print("[Jobs]\n", jobs.dtypes)
print("\n[Candidates]\n", candidates.dtypes)
print("\n[Train]\n", applications_train.dtypes)
print("\n")

print("--- 4. Label Distribution ---")
print(applications_train['relevance_label'].value_counts().sort_index())

--- 1. Unique Values ---
Currencies: <StringArray>
[  'USD',   'EUR',   'IRR', 'IRR  ',  'USD ',   'irr', 'USD  ',   'usd',
   'eur',  'EUR ',  'IRR ', 'EUR  ']
Length: 12, dtype: str
English Levels: <StringArray>
['Upper-Intermediate',                 'C2',                 'C1',
                 'B1',                 'B2',                 'A2',
             'Native',                 'A1',           'Beginner',
           'Advanced',       'Intermediate',         'Elementary',
             'Fluent',          'Bilingual']
Length: 14, dtype: str
Education Levels: <StringArray>
[        'PhD',    'Bachelor',      'Master', 'High School',        'B.A.',
        'B.S.',         'BSc',        'M.S.',        'M.A.',       'Ph.D.',
         'MSc',          'HS',   'Doctorate',   'Secondary']
Length: 14, dtype: str


--- 2. Missing Values ---
[Jobs]
 required_skills         248
min_years_experience    105
max_years_experience    495
remote_allowed          265
company_size            377
dtype:

In [4]:

jobs = pd.read_csv('jobs.csv')
candidates = pd.read_csv('candidates.csv')
applications_train = pd.read_csv('applications_train.csv')

print("--- 1. Applicants per Job ---")
applicants_count = applications_train.groupby('job_id').size()
print("Min applicants:", applicants_count.min())
print("Max applicants:", applicants_count.max())
print("Mean applicants:", applicants_count.mean())

print("\n--- 2. Date Ranges ---")
print("Job Posted Min/Max:", jobs['job_posted_date'].min(), "-", jobs['job_posted_date'].max())
print("Application Min/Max:", applications_train['application_date'].min(), "-", applications_train['application_date'].max())

print("\n--- 3. Top Locations ---")
print("Jobs Top 10:\n", jobs['job_location'].value_counts().head(10))
print("Candidates Top 10:\n", candidates['candidate_location'].value_counts().head(10))

--- 1. Applicants per Job ---
Min applicants: 1
Max applicants: 150
Mean applicants: 32.38941914371421

--- 2. Date Ranges ---
Job Posted Min/Max: 01/04/2023 - 31/01/2023
Application Min/Max: 2023-01-01 - 2024-06-30

--- 3. Top Locations ---
Jobs Top 10:
 job_location
Tabriz        225
Madrid        187
Singapore     184
Tokyo         182
Vancouver     181
Mashhad       180
Isfahan       180
Manchester    180
Toronto       179
Qom           178
Name: count, dtype: int64
Candidates Top 10:
 candidate_location
Dubai         1804
Amsterdam     1801
Boston        1796
Singapore     1788
Toronto       1784
Manchester    1778
Karaj         1760
Berlin        1748
Qom           1743
Paris         1742
Name: count, dtype: int64


In [5]:
import pandas as pd
import numpy as np

def run_phase_one_two(train_path, test_path, jobs_path, candidates_path):
    df_train = pd.read_csv(train_path)
    df_test = pd.read_csv(test_path)
    jobs = pd.read_csv(jobs_path)
    candidates = pd.read_csv(candidates_path)

    jobs['salary_currency'] = jobs['salary_currency'].astype(str).str.strip().str.upper()
    exchange_rates = {'USD': 1.0, 'EUR': 1.08, 'IRR': 0.000002}
    
    jobs['salary_min_usd'] = jobs.apply(lambda row: row['salary_min'] * exchange_rates.get(row['salary_currency'], 1.0) if pd.notnull(row['salary_min']) else np.nan, axis=1)
    jobs['salary_max_usd'] = jobs.apply(lambda row: row['salary_max'] * exchange_rates.get(row['salary_currency'], 1.0) if pd.notnull(row['salary_max']) else np.nan, axis=1)

    candidates['english_proficiency'] = candidates['english_proficiency'].astype(str).str.strip()
    eng_map = {'A1': 1, 'Beginner': 1, 'Elementary': 1, 'A2': 2, 'Pre-Intermediate': 2, 'B1': 3, 'Intermediate': 3, 'B2': 4, 'Upper-Intermediate': 4, 'C1': 5, 'Advanced': 5, 'C2': 6, 'Fluent': 6, 'Bilingual': 6, 'Native': 6}
    candidates['english_score'] = candidates['english_proficiency'].map(eng_map)

    candidates['education_level'] = candidates['education_level'].astype(str).str.strip()
    edu_map = {'High School': 1, 'HS': 1, 'Secondary': 1, 'Bachelor': 2, 'B.A.': 2, 'B.S.': 2, 'BSc': 2, 'Master': 3, 'M.S.': 3, 'M.A.': 3, 'MSc': 3, 'PhD': 4, 'Ph.D.': 4, 'Doctorate': 4}
    candidates['edu_score'] = candidates['education_level'].map(edu_map)

    jobs['job_location'] = jobs['job_location'].astype(str).str.lower().str.strip()
    candidates['candidate_location'] = candidates['candidate_location'].astype(str).str.lower().str.strip()

    jobs['job_posted_date'] = pd.to_datetime(jobs['job_posted_date'], errors='coerce')
    df_train['application_date'] = pd.to_datetime(df_train['application_date'], errors='coerce')
    df_test['application_date'] = pd.to_datetime(df_test['application_date'], errors='coerce')

    jobs['min_years_experience'] = jobs['min_years_experience'].fillna(0)
    jobs['max_years_experience'] = jobs['max_years_experience'].fillna(50)
    jobs['remote_allowed'] = jobs['remote_allowed'].fillna('No')
    
    candidates['years_experience'] = candidates['years_experience'].fillna(0)
    candidates['willing_to_relocate'] = candidates['willing_to_relocate'].fillna('No')
    candidates['english_score'] = candidates['english_score'].fillna(candidates['english_score'].median())
    candidates['edu_score'] = candidates['edu_score'].fillna(candidates['edu_score'].median())
    candidates['expected_salary'] = candidates['expected_salary'].fillna(candidates['expected_salary'].median())

    train_merged = df_train.merge(jobs, on='job_id', how='left').merge(candidates, on='candidate_id', how='left')
    test_merged = df_test.merge(jobs, on='job_id', how='left').merge(candidates, on='candidate_id', how='left')
    
    return train_merged, test_merged

train_data, test_data = run_phase_one_two('applications_train.csv', 'applications_test.csv', 'jobs.csv', 'candidates.csv')

print("=== Phase 1 & 2 Sanity Check ===")
print(f"Train Data Shape: {train_data.shape}")
print(f"Test Data Shape: {test_data.shape}")
print("\n[Sample Data Check - First 3 Rows]")
print(train_data[['application_id', 'salary_min_usd', 'english_score', 'edu_score', 'years_experience', 'relevance_label']].head(3))
print("\n[Missing Values in Key Columns]")
print(train_data[['job_id', 'candidate_id', 'salary_min_usd', 'english_score', 'edu_score']].isnull().sum())
print("================================")

=== Phase 1 & 2 Sanity Check ===
Train Data Shape: (118772, 34)
Test Data Shape: (52700, 33)

[Sample Data Check - First 3 Rows]
   application_id  salary_min_usd  english_score  edu_score  years_experience  \
0             206          5040.0            4.0          3               8.0   
1             207          5040.0            5.0          4               5.0   
2             208          5040.0            3.0          2               2.0   

   relevance_label  
0                2  
1                1  
2                2  

[Missing Values in Key Columns]
job_id            0
candidate_id      0
salary_min_usd    0
english_score     0
edu_score         0
dtype: int64


In [6]:
def run_phase_three(train_df, test_df):
    def engineer_features(df):
        # 1. Skill Match Ratio
        def get_skill_match(row):
            if pd.isna(row['required_skills']) or pd.isna(row['skills']):
                return 0.0
            req = set(row['required_skills'].split('|'))
            cand = set(row['skills'].split('|'))
            if len(req) == 0: 
                return 0.0
            return len(req.intersection(cand)) / len(req)
        
        df['skill_match_ratio'] = df.apply(get_skill_match, axis=1)

        # 2. Salary Gap
        df['job_avg_salary_usd'] = df[['salary_min_usd', 'salary_max_usd']].mean(axis=1)
        df['salary_gap'] = df['expected_salary'] - df['job_avg_salary_usd']
        df['salary_gap'] = df['salary_gap'].fillna(0) # پر کردن گپ‌های احتمالی

        # 3. Experience Gap
        df['experience_gap'] = df['years_experience'] - df['min_years_experience']

        # 4. Location Match
        df['location_match'] = ((df['candidate_location'] == df['job_location']) | 
                                (df['remote_allowed'].str.lower() == 'yes') | 
                                (df['willing_to_relocate'].str.lower() == 'yes')).astype(int)

        # 5. Application Speed (Days to Apply)
        df['days_to_apply'] = (df['application_date'] - df['job_posted_date']).dt.days
        df['days_to_apply'] = df['days_to_apply'].fillna(df['days_to_apply'].median())
        df.loc[df['days_to_apply'] < 0, 'days_to_apply'] = 0 # اصلاح باگ‌های زمانی منفی
        
        return df
    
    print("Engineering features for train data...")
    train_eng = engineer_features(train_df.copy())
    print("Engineering features for test data...")
    test_eng = engineer_features(test_df.copy())
    
    return train_eng, test_eng

# اجرای فاز سوم
train_data_eng, test_data_eng = run_phase_three(train_data, test_data)

# --- Sanity Check فاز 3 ---
print("\n=== Phase 3 Sanity Check ===")
features_to_check = ['skill_match_ratio', 'salary_gap', 'experience_gap', 'location_match', 'days_to_apply']
print("[Sample Data Check - New Features]")
print(train_data_eng[features_to_check + ['relevance_label']].head())
print("\n[Missing Values in New Features]")
print(train_data_eng[features_to_check].isnull().sum())
print("============================")

Engineering features for train data...
Engineering features for test data...

=== Phase 3 Sanity Check ===
[Sample Data Check - New Features]
   skill_match_ratio     salary_gap  experience_gap  location_match  \
0                0.2  110505.275330             4.0               0   
1                0.0   80816.039783             1.0               0   
2                0.6   85805.832040            -2.0               0   
3                0.0   53624.450025            -2.0               1   
4                0.0   45015.932462            -3.0               1   

   days_to_apply  relevance_label  
0           24.0                2  
1           14.0                1  
2           24.0                2  
3           24.0                1  
4           14.0                0  

[Missing Values in New Features]
skill_match_ratio    0
salary_gap           0
experience_gap       0
location_match       0
days_to_apply        0
dtype: int64


In [7]:
!pip install lightgbm scikit-learn

  Using cached threadpoolctl-3.6.0-py3-none-any.whl.metadata (13 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 97.2 kB/s  0:00:388.3 kB/s eta 0:00:02:24m
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 142.5 kB/s  0:00:58 eta 0:00:010:00:02
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.2/35.2 MB 210.6 kB/s  0:03:42 eta 0:00:010:00:11
Using cached threadpoolctl-3.6.0-py3-none-any.whl (18 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5/5 [lightgbm]━━ 4/5 [lightgbm]arn]


In [8]:
import lightgbm as lgb
from sklearn.model_selection import GroupKFold
import numpy as np
import pandas as pd

def run_phase_four(train_df):
    # ۱. حذف ستون‌های متنی، تاریخ‌ها و IDها (مدل فقط عدد می‌فهمد)
    cols_to_drop = [
        'application_id', 'candidate_id', 'application_date', 'job_posted_date',
        'job_title', 'required_skills', 'salary_currency', 'job_location', 'remote_allowed', 
        'company_size', 'industry', 'current_title', 'skills', 'education_level', 
        'university', 'previous_companies', 'certifications', 'english_proficiency', 
        'candidate_location', 'willing_to_relocate', 'account_created_date',
        'relevance_label' # لیبل هدف را نباید به عنوان فیچر به مدل بدهیم!
    ]
    
    # استخراج فیچرهای نهایی برای آموزش
    features = [c for c in train_df.columns if c not in cols_to_drop]
    
    X = train_df[features]
    y = train_df['relevance_label']
    groups = train_df['job_id']
    
    # ۲. تعریف GroupKFold برای ایزوله کردن شغل‌ها
    gkf = GroupKFold(n_splits=5)
    oof_predictions = np.zeros(len(train_df))
    
    # ۳. تعریف مدل رگرسیون LightGBM
    model = lgb.LGBMRegressor(
        n_estimators=200,
        learning_rate=0.05,
        max_depth=7,
        random_state=42,
        n_jobs=-1
    )
    
    # ذخیره اهمیت ویژگی‌ها برای تحلیل
    feature_importances = np.zeros(len(features))
    
    print(f"Training on {len(features)} features...")
    
    # ۴. حلقه آموزش و اعتبارسنجی
    for fold, (train_idx, val_idx) in enumerate(gkf.split(X, y, groups)):
        X_train, y_train = X.iloc[train_idx], y.iloc[train_idx]
        X_val, y_val = X.iloc[val_idx], y.iloc[val_idx]
        
        model.fit(
            X_train, y_train,
            eval_set=[(X_val, y_val)],
            callbacks=[lgb.early_stopping(stopping_rounds=30, verbose=False)]
        )
        
        # پیش‌بینی روی دیتای ولیدیشن
        oof_predictions[val_idx] = model.predict(X_val)
        feature_importances += model.feature_importances_ / gkf.n_splits
        print(f"Fold {fold+1} completed.")
        
    train_df['predicted_score'] = oof_predictions
    
    # ساخت دیتافریم اهمیت ویژگی‌ها
    fi_df = pd.DataFrame({
        'Feature': features,
        'Importance': feature_importances
    }).sort_values(by='Importance', ascending=False)
    
    return train_df, fi_df, model, features

# اجرای فاز ۴
train_data_scored, feature_importance_df, final_model, final_features = run_phase_four(train_data_eng)

# --- Sanity Check فاز 4 ---
print("\n=== Phase 4 Sanity Check ===")
print("[Top 10 Most Important Features]")
print(feature_importance_df.head(10))
print("\n[Sample Predictions vs Actual Labels]")
print(train_data_scored[['job_id', 'candidate_id', 'relevance_label', 'predicted_score']].head())
print("============================")

Training on 18 features...
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.023718 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 2241
[LightGBM] [Info] Number of data points in the train set: 95017, number of used features: 18
[LightGBM] [Info] Start training from score 1.148952
Fold 1 completed.
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.001844 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 2242
[LightGBM] [Info] Number of data points in the train set: 95017, number of used features: 18
[LightGBM] [Info] Start training from score 1.148752
Fold 2 completed.
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000804 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough

In [9]:

def calculate_metrics(df):
    ndcg_scores = []
    ap_scores = []
    
    for job_id, group in df.groupby('job_id'):
        y_true = group['relevance_label'].values
        y_pred = group['predicted_score'].values
        
        order = np.argsort(y_pred)[::-1]
        y_true_sorted = y_true[order]
        
        ideal_order = np.argsort(y_true)[::-1]
        y_ideal_sorted = y_true[ideal_order]
        
        def dcg_at_k(y, k):
            y_k = y[:k]
            gains = 2**y_k - 1
            discounts = np.log2(np.arange(len(y_k)) + 2)
            return np.sum(gains / discounts)
        
        dcg = dcg_at_k(y_true_sorted, 10)
        idcg = dcg_at_k(y_ideal_sorted, 10)
        
        if idcg == 0:
            ndcg_scores.append(0.0)
        else:
            ndcg_scores.append(dcg / idcg)
        
        y_true_binary = (y_true >= 3).astype(int)
        y_true_sorted_binary = (y_true_sorted >= 3).astype(int)
        
        total_relevant = np.sum(y_true_binary)
        
        if total_relevant == 0:
            ap_scores.append(0.0)
            continue
            
        y_k_bin = y_true_sorted_binary[:5]
        precisions = []
        rel_count = 0
        
        for i, rel in enumerate(y_k_bin):
            if rel == 1:
                rel_count += 1
                precisions.append(rel_count / (i + 1))
        
        if len(precisions) == 0:
            ap_scores.append(0.0)
        else:
            ap = np.sum(precisions) / min(total_relevant, 5)
            ap_scores.append(ap)
            
    return np.mean(ndcg_scores), np.mean(ap_scores)

mean_ndcg, mean_ap = calculate_metrics(train_data_scored)
print(f"Mean NDCG@10: {mean_ndcg:.4f}")
print(f"MAP@5: {mean_ap:.4f}")

Mean NDCG@10: 0.7731
MAP@5: 0.5504


In [10]:

def run_phase_six(test_df, model, features):
    print("Preparing test data for prediction...")
    
    # جدا کردن دقیقاً همان فیچرهایی که مدل با آن‌ها آموزش دیده است
    X_test = test_df[features]
    
    print("Generating predictions...")
    # پیش‌بینی امتیازها برای دیتای تست
    test_predictions = model.predict(X_test)
    
    # ساخت دیتافریم نهایی طبق استاندارد کوئرا
    submission_df = pd.DataFrame({
        'application_id': test_df['application_id'],
        'score': test_predictions
    })
    
    # ذخیره فایل CSV بدون ستون ایندکس
    submission_file_name = 'submission.csv'
    submission_df.to_csv(submission_file_name, index=False)
    
    return submission_df, submission_file_name

# اجرای فاز ۶
final_submission, file_name = run_phase_six(test_data_eng, final_model, final_features)

# --- Sanity Check فاز 6 ---
print("\n=== Phase 6 Sanity Check ===")
print(f"Submission File Shape: {final_submission.shape} (باید دقیقاً 52700 سطر و 2 ستون باشد)")
print("\n[Sample Submission - First 5 Rows]")
print(final_submission.head())
print("\n[Missing Values Check]")
print(final_submission.isnull().sum())
print(f"\n✅ File '{file_name}' successfully generated in your directory!")
print("============================")

Preparing test data for prediction...
Generating predictions...

=== Phase 6 Sanity Check ===
Submission File Shape: (52700, 2) (باید دقیقاً 52700 سطر و 2 ستون باشد)

[Sample Submission - First 5 Rows]
   application_id     score
0               1  2.504817
1               2  1.960901
2               3  1.028975
3               4  1.319815
4               5  1.406372

[Missing Values Check]
application_id    0
score             0
dtype: int64

✅ File 'submission.csv' successfully generated in your directory!
